#Import libraries

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


##Data ingestions 
###Understand the four tables

In [0]:
#loading the customers table from Databricks into a Spark DataFrame and then store it in the variable customer
#Spark:Conceptually, it functions exactly like a table in a relational database or a spreadsheet, but with one massive difference: while a regular spreadsheet sits on a single computer, a Spark DataFrame is split across a cluster of thousands of computers, allowing it to process massive big data workloads in parallel.
customer=spark.table("workspace.default.customers")
customers=customer.toPandas()
display(customers)




In [0]:
#loading the orders table from Databricks into a Spark DataFrame and then store it in the variable orders
orders=spark.table("workspace.default.orders")
orders=orders.toPandas()
display(orders)

In [0]:
#loading the payments table from Databricks into a Spark DataFrame and then store it in the variable payments
payments=spark.table("workspace.default.payments")
payments=payments.toPandas()
display(payments)

In [0]:
#loading the products table from Databricks into a Spark DataFrame and then store it in the variable products
products=spark.table("workspace.default.products")
products=products.toPandas()
display(payments)

- Key findings: We have 4 levels of information
- Customers represents  the customer Customer ID, AGE, signupdate, city and segment 
- Orders represents the 
- Payments
- Products

#Exploratory Data analysis

In [0]:
customers.shape
#The .shape tells us the dimensions of the table we have 1000 rows and 5 columns for the customers table. 

In [0]:
orders.shape
#The .shape tells us that the we have 50120 rows, and 8 columns for the orders table

In [0]:
payments.shape
#The .shape tells us that we have 50000 rows and 4 columns for the payments table


In [0]:
products.shape
#The .shape tells us that we have 20 rows and 4 columns for the products table

In [0]:
#check the columns found in the customers table and help state the data types for each column
#check what data fields are available before you perform data analysis, filtering, or calculations.
customers.columns

In [0]:
#Check the columns found in the orders table 
orders.columns

In [0]:
#Check the columns found in the products table 
products.columns

##Orders Section

In [0]:
#Check the columns found in the payments table
payments.columns

In [0]:
#structural summary of a DataFrame from the customers table we see that the customerID is in the correct formating as an integer
#Age is reflecting the correct data type as a float64 ! this is flagged as age cannot be a float
#City is a string object works well
#SignupDate is a date object we can convert it to date and time 
#CustomerSegment is a string object we can make it a category object
customers.info()

In [0]:
#Structural summary of DataFrame from the orders table
#ordersID is in the correct data type as int64
#CustomerID is in the correct data type as int64
#Orderdate is an object and might need to be converted to a data
#ProductID is in the correct format
#Quantity is in the correct format
#Discount is in the correct format
#Payment Method might need to be converted to string but will decide
#Status is an object
orders.info()

In [0]:
#PaymentsID is in the correct format
#OrderID is in the correct format
#PaymentDate is an object and might need to be converted to a date
#PaymentStatus is an object

payments.info()

In [0]:
#Products is in the correct data type
#Products is in the correct data type
#Catergory we need to decide whether we need to convert it.
#UnitPrice is in the correct data type
products.info()
#check the data types for the products table


#Check for data Quality

- Outcome for this section 
- Find missing value
- Incorrect data types(we really looking into this)
- Invalid values
- Inconsistent categories 


In [0]:
#Number of nulls in the customers table
#There is 180 nulls in the Age columns and 119 in the City columns 
customers.isnull().sum()

In [0]:
#Number of nulls that are duplicated in the customers table
#No duplicates in the customers table
customers.duplicated().sum()

In [0]:
#statistical summary of a dataset in the customers data

customer.describe()

In [0]:
#OrderDate has 35 null value 
#Quantity has 80 null values
#Discount has 221 Null value
#PaymentMethod 452
#Data type int64
orders.isnull().sum()

In [0]:
#We have 120 duplicated values
orders.duplicated().sum()

In [0]:
orders.describe().round(2)
#Pandas is used for data manipulation and analysis so below is a satistical summary of the orders table

In [0]:
#No null 
products.isnull().sum()

In [0]:
#There is no duplicated values in the columns or rows
products.duplicated().sum()

In [0]:
#For me it made sense to use the statistical representation of the unit price then the other columns 
products.describe()["UnitPrice"].round(2)

In [0]:
#PaymentDate date has 35 null values
payments.isnull().sum()

In [0]:
#No duplicated data found in this table
payments.duplicated().sum()


In [0]:
#No need for the Describe, this table is used 

#EDA Analyse individual variable

In [0]:
customers.columns


In [0]:
#Find the inconsistent values
customers["City"].value_counts()

In [0]:
customers["City"].unique()

In [0]:
#Trying to Upper case the city, and remove any unwanted spaces and correct spelling.
customers["City"] = customers["City"].astype(str).str.strip().str.title().replace({"Mashad": "Mashhad"})

In [0]:
orders.columns

In [0]:
#Different number of segment
customers["CustomerSegment"].value_counts()


In [0]:
#Different number of ages 
customers["Age"].value_counts()

In [0]:
#Correcting the date data to datetime 
customers["SignupDate"] = pd.to_datetime(customers["SignupDate"])
customers["Age"] = customers["Age"].astype("Int64")

In [0]:
orders.columns
#the columns shown here for better overview of the data


In [0]:
# There are 3 types status in the orders table namely completed, cancelled, Returned
#Completed: 46105
#Cancelled: 2477
#Returned: 1538
orders["Status"].value_counts()


In [0]:
orders["Discount"].value_counts()

In [0]:
#Remove any unwanted spaces and Upper case the 
orders["PaymentMethod"] = orders["PaymentMethod"].astype(str).str.strip().str.title()

In [0]:
payments.columns

In [0]:
payments["PaymentStatus"].value_counts()

#There are 3 types of payment status in the payments table namely completed, failed, and refunded
#We have 46569 completed payments,1924 Failed, and 1507 Refunded

In [0]:
#Remove any unwanted spaces and Upper case the first letter
payments["PaymentStatus"] = payments["PaymentStatus"].astype(str).str.strip().str.title()

In [0]:
products.columns

In [0]:
#Product category 6 categories. 
products["Category"].value_counts()


In [0]:
#Correct the caps for the ProductName and Category.Remove any unwanted spaces for both columns
products["ProductName"] = products["ProductName"].astype(str).str.strip().str.title()
products["Category"] = products["Category"].astype(str).str.strip().str.title()

#Merge tables

In [0]:
df = orders.merge(payments[["OrderID", "PaymentStatus"]], on="OrderID", how="left")
print(len(orders), len(df))

In [0]:
df = df.merge(products, on="ProductID", how="left")
print(len(df))   # should still match

In [0]:
df = df.merge(customers[["CustomerID", "City", "CustomerSegment"]], on="CustomerID", how="left")
print(len(df))   # should still match

1. Revenue

In [0]:
df["Revenue"] = df["Quantity"] * df["UnitPrice"] * (1 - df["Discount"])


2 Year and Month (for looking at trends over time)

In [0]:
df["OrderDate"] = pd.to_datetime(df["OrderDate"])
df["Year"] = df["OrderDate"].dt.year
df["Month"] = df["OrderDate"].dt.to_period("M")

So the 80 orders have missing quantity values from the script above. Only 72 are completed orders, 7 are cancelled, and 1 is returned. Since most missing quantities occur in completed orders, the values should not automatically be assumed to represent zero. The records should be retained and the missing quantities investigated before calculating sales